# Importing required LIbraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # ⚠️ Required to enable it
from sklearn.impute import IterativeImputer


# Creating a dictionary

In [2]:
students = {
    "S101": {
        "Name": "Ahmed",
        "Age": 20,
        "Courses": {
            "Math": 85, "Science": 90, "English": 78
        },
        "Address": {
            "Street": "123 Main Street",
            "City": "Islamabad",
            "State": "CA",
            "ZIP": "12345"
        }
    },
    "S102": {
        "Name": "Bilal",
        "Age": None,  # Missing Age
        "Courses": {
            "Math": 95, "Science": None, "English": 58
        },
        "Address": {
            "Street": None,
            "City": "Islamabad",
            "State": "CA",
            "ZIP": "051"
        }
    },
    "S103": {
        "Name": "Saim",
        "Age": 20,
        "Courses": {
            "Math": 55, "Science": 77, "English": None
        },
        # Missing Address entirely
    },
    "S104": {
        "Name": "Zohaib",
        "Age": 20,
        "Courses": {
            "Math": 95, "Science": 90, "English": 78
        },
        "Address": {
            "Street": "45 Adiala Main Street",
            "City": None,
            "State": "CA",
            "ZIP": "051"
        }
    },
    "S105": {
        "Name": None,
        "Age": 23,
        # Missing Courses entirely
        "Address": {
            "Street": "78 Falcon Street",
            "City": "Islamabad",
            "State": None,
            "ZIP": "055"
        }
    }
}


# Coverting to dataframe

In [3]:
flattened_students = []

for sid, info in students.items():
    base_info = {
        "StudentID": sid,
        "Name": info.get("Name"),
        "Age": info.get("Age"),
    }
    
    
    courses = info.get("Courses", {})
    base_info["Math"] = courses.get("Math")
    base_info["Science"] = courses.get("Science")
    base_info["English"] = courses.get("English")
    
    
    address = info.get("Address", {})
    base_info["Street"] = address.get("Street")
    base_info["City"] = address.get("City")
    base_info["State"] = address.get("State")
    base_info["ZIP"] = address.get("ZIP")
    
    flattened_students.append(base_info)


df = pd.DataFrame(flattened_students)

df.set_index('StudentID', inplace=True)


print(df)


             Name   Age  Math  Science  English                 Street  \
StudentID                                                                
S101        Ahmed  20.0  85.0     90.0     78.0        123 Main Street   
S102        Bilal   NaN  95.0      NaN     58.0                   None   
S103         Saim  20.0  55.0     77.0      NaN                   None   
S104       Zohaib  20.0  95.0     90.0     78.0  45 Adiala Main Street   
S105         None  23.0   NaN      NaN      NaN       78 Falcon Street   

                City State    ZIP  
StudentID                          
S101       Islamabad    CA  12345  
S102       Islamabad    CA    051  
S103            None  None   None  
S104            None    CA    051  
S105       Islamabad  None    055  


# Applying Imputer (ITERATIVE)

## Seperating Numeric Columns

In [4]:
numeric_df = df.select_dtypes(include=['float64', 'int64'])

## Applying Imputer

In [5]:
iter_imputer = IterativeImputer(max_iter=10, random_state=0)
imputed_array = iter_imputer.fit_transform(numeric_df)


c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


## Converting back to dataframe

In [6]:
imputed_df = pd.DataFrame(imputed_array, columns=numeric_df.columns, index=df.index)


## Merging 

In [7]:
final_df = df.copy()
final_df[numeric_df.columns] = imputed_df


# Printing dataframe

In [8]:
print(final_df)


             Name      Age       Math    Science    English  \
StudentID                                                     
S101        Ahmed  20.0000  85.000000  90.000000  78.000000   
S102        Bilal  31.9999  95.000000  96.947574  58.000000   
S103         Saim  20.0000  55.000000  77.000000  87.395138   
S104       Zohaib  20.0000  95.000000  90.000000  78.000000   
S105         None  23.0000  82.500008  88.486906  75.348744   

                          Street       City State    ZIP  
StudentID                                                 
S101             123 Main Street  Islamabad    CA  12345  
S102                        None  Islamabad    CA    051  
S103                        None       None  None   None  
S104       45 Adiala Main Street       None    CA    051  
S105            78 Falcon Street  Islamabad  None    055  
